# 01 — Validate parsed holdings

Parse the latest Stashaway statement, apply role tags from `book_config`,
and check that every parsed portfolio reconciles cleanly against the
weight-sum tolerance and resolves against the universe map.

**Inputs:** `data/statements/<latest>.pdf`

**Outputs:** a summary table; a per-portfolio holdings table; a list of any
unmapped tickers that need adding to `STASHAWAY_UNIVERSE`.


In [ ]:
from pathlib import Path
import pandas as pd

from hailmary.allocation.book_config import ROLES
from hailmary.allocation.portfolios import Role, from_parsed
from hailmary.allocation.statements import parse_statement

STATEMENT_PATH = Path('../../data/statements/2026-04 StashAway Monthly Statement.pdf')
assert STATEMENT_PATH.exists(), f'Statement not found at {STATEMENT_PATH}'

## Parse the statement

In [ ]:
parsed = parse_statement(STATEMENT_PATH, use_cache=False)
print(f'Parsed {len(parsed)} portfolios from {STATEMENT_PATH.name}')
print(f'Statement date: {parsed[0].statement_date}')

## Reconcile weights

Each portfolio's weights should sum to 1.0 ± 1e-4. If any fail, the parser
raised inside `parse_statement`; this cell is a final visual check.

In [ ]:
rows = [
    {
        'name': pf.name,
        'currency': pf.currency,
        'total_value': pf.total_value,
        'n_holdings': len(pf.holdings),
        'weight_sum': sum(h.weight for h in pf.holdings),
    }
    for pf in parsed
]
summary = pd.DataFrame(rows).sort_values('total_value', ascending=False)
summary

## Apply role tags and resolve universe

`from_parsed` looks each holding up in `STASHAWAY_UNIVERSE`. An
`UnknownAssetError` here means a ticker needs adding to the map.

In [ ]:
portfolios = []
missing_roles = []
for pf in parsed:
    roles = ROLES.get(pf.name)
    if roles is None:
        missing_roles.append(pf.name)
        continue
    portfolios.append(from_parsed(pf, roles=roles))

if missing_roles:
    print('Portfolios with no role tag (edit book_config.ROLES):')
    for n in missing_roles:
        print(f'  - {n!r}')
else:
    print(f'All {len(portfolios)} portfolios resolved against the universe map.')

## Diagnostic vs hidden portfolios

In [ ]:
diag_rows = [
    {
        'name': p.name,
        'roles': ','.join(sorted(r.value for r in p.roles)),
        'currency': p.currency,
        'total_value': p.total_value,
        'n_holdings': len(p.holdings),
    }
    for p in portfolios
]
diag = pd.DataFrame(diag_rows)
in_diag = diag[diag['roles'].str.contains('holding')]
hidden = diag[~diag['roles'].str.contains('holding')]
print(f'In diagnostic ({len(in_diag)}):')
display(in_diag.sort_values('total_value', ascending=False))
print(f'Hidden ({len(hidden)}):')
display(hidden)

## Detailed holdings — first three portfolios

In [ ]:
for p in portfolios[:3]:
    print(f'\n=== {p.name} ({p.currency}) — total {p.total_value:,.2f} ===')
    rows = [
        {
            'stashaway_id': h.stashaway_id,
            'yahoo_ticker': h.metadata.ticker,
            'asset_class': h.metadata.asset_class,
            'region': h.metadata.region,
            'sector': h.metadata.sector,
            'weight': h.weight,
            'value': h.value,
        }
        for h in p.holdings
    ]
    display(pd.DataFrame(rows))

## Final assertion

If any of these fail, the rest of the diagnostic pipeline can't trust the
input — fix before running notebooks 02 and 03.

In [ ]:
from hailmary.allocation.portfolios import Role

assert len(portfolios) == 15, f'Expected 15 portfolios, got {len(portfolios)}'
for p in portfolios:
    assert abs(sum(h.weight for h in p.holdings) - 1.0) < 1e-4, p.name
managed = [p for p in portfolios if Role.MANAGED_BENCHMARK in p.roles]
assert len(managed) >= 1, 'No MANAGED_BENCHMARK portfolios — diagnostic will skip benchmark deltas'
print('All checks passed — proceed to notebook 02 / 03.')